# Assignment 5: Perceptron
**Both Part 1 (Heuristic) and Part 2 (Gradient Descent)**

## Data Loading
Load and plot data from `data.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Load data
df = pd.read_csv('data.csv', header=None, names=['x1', 'x2', 'label'])
X = df[['x1', 'x2']].values
y = df['label'].values

print(f'Dataset shape: {X.shape}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')
print(df.head())

In [ ]:
def plot_data(X, y, title='Data Plot'):
    """Plot the dataset with two classes."""
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(X[y==1, 0], X[y==1, 1], color='blue', label='Class 1', s=40)
    ax.scatter(X[y==0, 0], X[y==0, 1], color='red',  label='Class 0', s=40)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_data(X, y, title='Data from data.csv')

---
## Part 1: Heuristic Perceptron

Uses binary step activation (ŷ ∈ {0, 1}).
- Start with random weights and bias
- For each misclassified point: update weights and bias
- Plot separation lines: **red** (initial), **dashed green** (each iteration), **black** (final)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def step(z):
    """Binary step function for heuristic perceptron."""
    return (z >= 0).astype(int)

def get_line_points(w, b, x_range=(0, 1)):
    """
    Compute two y-points for the decision boundary line:
    w[0]*x1 + w[1]*x2 + b = 0  =>  x2 = -(w[0]*x1 + b) / w[1]
    """
    xs = np.array(x_range)
    if abs(w[1]) < 1e-10:
        return None
    ys = -(w[0] * xs + b) / w[1]
    return xs, ys


class HeuristicPerceptron:
    """
    Heuristic (classic) Perceptron with binary step classification.
    Update rule (for misclassified point (xi, yi)):
      - b  <- b  + r
      - wi <- wi + r * xi  (for all i)
    """
    def __init__(self, learning_rate=0.1, n_epochs=100, random_state=42):
        self.lr = learning_rate
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n_samples, n_features = X.shape
        self.w = rng.randn(n_features)
        self.b = rng.randn()
        self.history = [(self.w.copy(), self.b)]   # store (w, b) per iteration

        for epoch in range(self.n_epochs):
            misclassified = 0
            for xi, yi in zip(X, y):
                z = np.dot(self.w, xi) + self.b
                y_hat = step(z)
                if y_hat != yi:
                    misclassified += 1
                    # Update rule: move toward correct class
                    r = self.lr if yi == 1 else -self.lr
                    self.b += r
                    self.w += r * xi
                    self.history.append((self.w.copy(), self.b))
            if misclassified == 0:
                print(f'Converged at epoch {epoch+1}')
                break
        return self

    def predict(self, X):
        z = X.dot(self.w) + self.b
        return step(z)

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

In [ ]:
def plot_heuristic_boundaries(X, y, perceptron, title='Solution boundary (Heuristic)'):
    """
    Plot all decision boundaries:
      - Red:          initial line (before any training)
      - Dashed green: all intermediate lines
      - Black solid:  final line
    """
    fig, ax = plt.subplots(figsize=(7, 6))

    # Scatter data
    ax.scatter(X[y==1, 0], X[y==1, 1], color='blue', s=30, zorder=5)
    ax.scatter(X[y==0, 0], X[y==0, 1], color='red',  s=30, zorder=5)

    x_range = (X[:, 0].min() - 0.05, X[:, 0].max() + 0.05)
    history = perceptron.history

    # Initial line - red
    res = get_line_points(history[0][0], history[0][1], x_range)
    if res:
        ax.plot(res[0], res[1], color='red', linewidth=1.5, label='Initial')

    # Intermediate lines - dashed green
    for w, b in history[1:-1]:
        res = get_line_points(w, b, x_range)
        if res:
            ax.plot(res[0], res[1], color='green', linestyle='--', linewidth=0.6, alpha=0.6)

    # Final line - black
    if len(history) > 1:
        res = get_line_points(history[-1][0], history[-1][1], x_range)
        if res:
            ax.plot(res[0], res[1], color='black', linewidth=2, label='Final')

    ax.set_xlim(x_range)
    ax.set_ylim(X[:, 1].min() - 0.1, X[:, 1].max() + 0.1)
    ax.set_title(title)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')

    green_patch = mlines.Line2D([], [], color='green', linestyle='--', label='Iterations')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [green_patch])
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Part 1: Train with default learning rate ---
hp = HeuristicPerceptron(learning_rate=0.1, n_epochs=200, random_state=42)
hp.fit(X, y)
print(f'Training Accuracy: {hp.accuracy(X, y)*100:.1f}%')
print(f'Total weight updates: {len(hp.history)-1}')
plot_heuristic_boundaries(X, y, hp, title='Solution boundary — Heuristic Perceptron (lr=0.1)')

In [ ]:
# --- Experiment: Play with learning rate (Part 1) ---
learning_rates = [0.01, 0.1, 1.0]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, lr in zip(axes, learning_rates):
    hp_exp = HeuristicPerceptron(learning_rate=lr, n_epochs=200, random_state=42)
    hp_exp.fit(X, y)

    ax.scatter(X[y==1, 0], X[y==1, 1], color='blue', s=25, zorder=5)
    ax.scatter(X[y==0, 0], X[y==0, 1], color='red',  s=25, zorder=5)

    x_range = (X[:, 0].min() - 0.05, X[:, 0].max() + 0.05)
    history = hp_exp.history

    res = get_line_points(history[0][0], history[0][1], x_range)
    if res: ax.plot(res[0], res[1], color='red', linewidth=1.5)

    for w, b in history[1:-1]:
        res = get_line_points(w, b, x_range)
        if res: ax.plot(res[0], res[1], color='green', linestyle='--', linewidth=0.5, alpha=0.5)

    if len(history) > 1:
        res = get_line_points(history[-1][0], history[-1][1], x_range)
        if res: ax.plot(res[0], res[1], color='black', linewidth=2)

    ax.set_xlim(x_range)
    ax.set_ylim(X[:, 1].min() - 0.1, X[:, 1].max() + 0.1)
    ax.set_title(f'Solution boundary\nlearning_rate={lr}  |  updates={len(history)-1}')
    ax.set_xlabel('x1'); ax.set_ylabel('x2')

plt.suptitle('Part 1 — Effect of Learning Rate (Heuristic Perceptron)', fontsize=14)
plt.tight_layout()
plt.show()

### Part 1 Analysis

- **Learning rate = 0.01**: Converges slowly; many small boundary adjustments are visible as tightly packed green dashed lines. The model is conservative and stable.
- **Learning rate = 0.1**: A balanced choice - converges in a moderate number of updates with clearly visible boundary evolution.
- **Learning rate = 1.0**: Large jumps per update; the boundary oscillates widely before (or without) settling. Risk of overshooting the solution.

The heuristic perceptron is guaranteed to converge **only if the data is linearly separable**. This dataset is approximately linearly separable, so convergence is achievable at smaller learning rates.

---
## Part 2: Gradient Descent Perceptron

Uses **sigmoid** activation: ŷ = σ(WX + b), so ŷ ∈ (0, 1) - continuous.
- Loss: **Log loss** = −[y·log(ŷ) + (1−y)·log(1−ŷ)]
- Update rule:
  - b  ← b  + r·(y − ŷ)
  - wᵢ ← wᵢ + r·(y − ŷ)·xᵢ
- Plot error every 10 epochs

In [ ]:
class GradientDescentPerceptron:
    """
    Perceptron trained with Gradient Descent.
    Activation: sigmoid  =>  ŷ ∈ (0,1) continuous.
    Loss: binary cross-entropy (log loss).
    Update rule per sample:
      b  <- b  + r*(y - ŷ)
      wi <- wi + r*(y - ŷ)*xi
    """
    def __init__(self, learning_rate=0.1, n_epochs=100, random_state=42):
        self.lr = learning_rate
        self.n_epochs = n_epochs
        self.random_state = random_state

    def _log_loss(self, X, y):
        z = X.dot(self.w) + self.b
        y_hat = sigmoid(z)
        eps = 1e-15
        y_hat = np.clip(y_hat, eps, 1 - eps)
        return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n_samples, n_features = X.shape
        self.w = rng.randn(n_features)
        self.b = rng.randn()
        self.history = [(self.w.copy(), self.b)]   # for boundary plotting
        self.errors = []                            # log loss per epoch
        self.error_epochs = []                      # epoch indices for plotting

        for epoch in range(self.n_epochs):
            for xi, yi in zip(X, y):
                z = np.dot(self.w, xi) + self.b
                y_hat = sigmoid(z)
                diff = yi - y_hat
                self.b += self.lr * diff
                self.w += self.lr * diff * xi

            self.history.append((self.w.copy(), self.b))

            # Record error every 10 epochs
            if (epoch + 1) % 10 == 0:
                loss = self._log_loss(X, y)
                self.errors.append(loss)
                self.error_epochs.append(epoch + 1)

        return self

    def predict_proba(self, X):
        z = X.dot(self.w) + self.b
        return sigmoid(z)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

In [ ]:
def plot_gd_boundaries(X, y, model, title='Solution boundary (GD Perceptron)'):
    """
    Plot decision boundaries for gradient descent perceptron.
    Colors: red=initial, dashed green=iterations, black=final.
    Also fills regions with class colors.
    """
    fig, ax = plt.subplots(figsize=(7, 6))

    # Decision region shading
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min()-0.1, X[:, 0].max()+0.1, 300),
        np.linspace(X[:, 1].min()-0.1, X[:, 1].max()+0.1, 300)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    proba = sigmoid(grid.dot(model.w) + model.b).reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=[0, 0.5, 1],
                colors=['#ffcccc', '#cce0ff'], alpha=0.3)

    # Scatter data
    ax.scatter(X[y==1, 0], X[y==1, 1], color='blue', s=30, zorder=5)
    ax.scatter(X[y==0, 0], X[y==0, 1], color='red',  s=30, zorder=5)

    x_range = (X[:, 0].min() - 0.05, X[:, 0].max() + 0.05)
    history = model.history

    # Initial - red
    res = get_line_points(history[0][0], history[0][1], x_range)
    if res: ax.plot(res[0], res[1], color='red', linewidth=1.5, label='Initial')

    # Intermediate - dashed green (subsample to keep plot clean)
    step_every = max(1, len(history) // 20)
    for w, b in history[1:-1:step_every]:
        res = get_line_points(w, b, x_range)
        if res: ax.plot(res[0], res[1], color='green', linestyle='--',
                        linewidth=0.7, alpha=0.6)

    # Final - black
    res = get_line_points(history[-1][0], history[-1][1], x_range)
    if res: ax.plot(res[0], res[1], color='black', linewidth=2, label='Final')

    ax.set_xlim(x_range)
    ax.set_ylim(X[:, 1].min() - 0.1, X[:, 1].max() + 0.1)
    ax.set_title(title)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    green_patch = mlines.Line2D([], [], color='green', linestyle='--', label='Iterations')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [green_patch])
    plt.tight_layout()
    plt.show()


def plot_error_curve(model, title='Error Plot'):
    """Plot log loss every 10 epochs."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(model.error_epochs, model.errors, color='steelblue', linewidth=2)
    ax.set_xlabel('Number of epochs')
    ax.set_ylabel('Error')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Part 2: Train Gradient Descent Perceptron ---
gd = GradientDescentPerceptron(learning_rate=0.1, n_epochs=100, random_state=42)
gd.fit(X, y)
print(f'Training Accuracy: {gd.accuracy(X, y)*100:.1f}%')
print(f'Final Log Loss: {gd.errors[-1]:.4f}')

plot_gd_boundaries(X, y, gd,
    title='Solution boundary - GD Perceptron (lr=0.1, epochs=100)')

plot_error_curve(gd, title='Error Plot - GD Perceptron (lr=0.1)')

In [ ]:
# --- Experiment: Play with learning rate and number of epochs (Part 2) ---
configs = [
    {'lr': 0.01,  'epochs': 100, 'label': 'lr=0.01, ep=100'},
    {'lr': 0.1,   'epochs': 100, 'label': 'lr=0.1,  ep=100'},
    {'lr': 1.0,   'epochs': 100, 'label': 'lr=1.0,  ep=100'},
    {'lr': 0.1,   'epochs': 50,  'label': 'lr=0.1,  ep=50'},
    {'lr': 0.1,   'epochs': 200, 'label': 'lr=0.1,  ep=200'},
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, cfg in enumerate(configs):
    ax = axes[i]
    m = GradientDescentPerceptron(learning_rate=cfg['lr'],
                                   n_epochs=cfg['epochs'],
                                   random_state=42)
    m.fit(X, y)

    ax.plot(m.error_epochs, m.errors, color='steelblue', linewidth=2)
    ax.set_xlabel('Epochs'); ax.set_ylabel('Log Loss')
    ax.set_title(f'Error — {cfg["label"]}\nFinal acc={m.accuracy(X,y)*100:.1f}%')
    ax.grid(True, alpha=0.3)

axes[-1].axis('off')
plt.suptitle('Part 2 — Effect of Learning Rate & Epochs (GD Perceptron)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Learning rate comparison: boundary plots (3 panels) ---
lrs = [0.01, 0.1, 1.0]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, lr in zip(axes, lrs):
    m = GradientDescentPerceptron(learning_rate=lr, n_epochs=100, random_state=42)
    m.fit(X, y)

    # Region shading
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min()-0.05, X[:, 0].max()+0.05, 200),
        np.linspace(X[:, 1].min()-0.05, X[:, 1].max()+0.05, 200)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    proba = sigmoid(grid.dot(m.w) + m.b).reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=[0, 0.5, 1],
                colors=['#ffcccc', '#cce0ff'], alpha=0.3)

    ax.scatter(X[y==1, 0], X[y==1, 1], color='blue', s=25, zorder=5)
    ax.scatter(X[y==0, 0], X[y==0, 1], color='red',  s=25, zorder=5)

    x_range = (X[:, 0].min()-0.05, X[:, 0].max()+0.05)
    history = m.history

    res = get_line_points(history[0][0], history[0][1], x_range)
    if res: ax.plot(res[0], res[1], color='red', linewidth=1.5)

    step_e = max(1, len(history) // 15)
    for w, b in history[1:-1:step_e]:
        res = get_line_points(w, b, x_range)
        if res: ax.plot(res[0], res[1], color='green', linestyle='--',
                        linewidth=0.7, alpha=0.6)

    res = get_line_points(history[-1][0], history[-1][1], x_range)
    if res: ax.plot(res[0], res[1], color='black', linewidth=2)

    ax.set_xlim(x_range)
    ax.set_ylim(X[:, 1].min()-0.1, X[:, 1].max()+0.1)
    ax.set_title(f'Solution boundary\nlr={lr}, epochs=100, acc={m.accuracy(X,y)*100:.1f}%')
    ax.set_xlabel('x1'); ax.set_ylabel('x2')

plt.suptitle('Part 2 — Learning Rate Comparison (GD Perceptron)', fontsize=14)
plt.tight_layout()
plt.show()

### Part 2 Analysis

**Effect of learning rate:**
- **lr = 0.01**: Very slow convergence; the error decreases smoothly but is still high after 100 epochs. Many more epochs would be needed.
- **lr = 0.1**: A good balance. The error decreases quickly and the boundary stabilizes within ~50-60 epochs.
- **lr = 1.0**: The error drops very fast initially, sometimes causing overshooting or instability. For some problems this can diverge.

**Effect of number of epochs:**
- More epochs allow the model to refine the boundary further. At `lr=0.1`, around 100 epochs is sufficient for this dataset.
- Diminishing returns are visible beyond 100 epochs — the error curve flattens, meaning the model has converged.

**Key difference from Part 1:**  
The gradient descent perceptron uses sigmoid (continuous output) instead of a binary step. This makes it differentiable, allowing gradient-based optimization. The log loss error plot shows a smooth, monotonically decreasing curve — a hallmark of gradient descent on a convex loss surface.

**Conclusion:** The GD perceptron with `lr=0.1` and `n_epochs=100` achieves high accuracy on this dataset. The decision boundary cleanly separates the two classes, consistent with the visual structure of the data.